In [23]:
# pip install pandas requests tqdm wandb ollama

In [24]:
import pandas as pd
import requests
import time
from ollama import ChatResponse
from tqdm import tqdm
import wandb

In [25]:
test_trans = pd.read_csv("dev_tran_ur.csv", header=None)
test_desc = pd.read_csv("dev_desc_ur.csv", header=None)

test_trans.columns = ["transcription"]
test_desc.columns = ["description"]


merged_df = pd.concat([test_trans, test_desc], axis=1)

# Display the first few rows of the merged DataFrame
merged_df.head()

,transcription,description
0,ہیلو ، میں سکاٹ ریل ہوں اور expertvillage.com ...,بارہماسی باغبانی پر اس مفت ویڈیو کلپ میں اپنے ...
1,ایلیسن کلنگر: expertvillage.com کی جانب سے ، م...,کافی مفت یا گھنے شیشے اس مفت ویڈیو سیریز میں د...
2,ٹھیک ہے تو یہ ہے بیومونٹ راگ ، پہلا نصف جو کہ ...,beaumont rag &#39;&#39; میں گانے کے حصے یا حصے...
3,ہم جھاڑو ہٹانے جا رہے ہیں اور آپ ہمیشہ اپنے کل...,چہرے کی سکرب کریم کو آہستہ سے دھو لیں اور جلد ...
4,آپ جانتے ہیں ، پچھلے حصے میں ، ہم نے یہ تسلیم ...,جب ایک ہی درجہ بندی والے کارڈ ایک دوسرے کے قری...


In [30]:
import pandas as pd
from ollama import Client
import time
from tqdm import tqdm  # Import tqdm for the progress bar

# Initialize the Ollama client with your settings
client = Client(
    host='http://localhost:11434',  # Replace with your Ollama API endpoint if different
    headers={'x-some-header': 'some-value'}  # Replace with your actual headers if necessary
)

# Function to get the summary using Ollama and an optional prompt
def summarize_transcription_with_ollama(transcription, model_name="llama3", prompt=None):
    try:
        # If no custom prompt is provided, use a default prompt for summarization
        if prompt is None:
            prompt = "Summarize this transcription as youtube description in Urdu:"
        
        # Prepare the final content for the message
        input_content = f"{prompt} {transcription}"  # Combine prompt with the transcription

        # Send the transcription to the Ollama model with the prompt
        response = client.chat(
            model=model_name,  # Specify the model
            messages=[{
                'role': 'user',
                'content': input_content  # Send the prompt + transcription as the user's message
            }]
        )
        
        # Extract and return the summary from the response
        return response['message']['content']  # You can also access it with response.message.content
    
    except Exception as e:
        print(f"Error: {e}")
        return "Error: Failed to summarize"

# Sample data (replace with your actual DataFrame)
test_trans = pd.read_csv("dev_tran_ur.csv", header=None)
test_desc = pd.read_csv("dev_desc_ur.csv", header=None)

# Rename columns for clarity
test_trans.columns = ["transcription"]
test_desc.columns = ["description"]

# Merge the transcriptions and descriptions into a single DataFrame
merged_df = pd.concat([test_trans, test_desc], axis=1)

# Optionally, define a custom prompt
custom_prompt = "Provide a short summary of this transcription."

# Define the model to be used (this can change for each run if needed)
model_name = "llama3"

# Apply the summarization function to each transcription and use tqdm for progress bar
summaries = []
for i, transcription in tqdm(enumerate(merged_df["transcription"]), total=len(merged_df), desc="Summarizing", unit="transcription"):
    summary = summarize_transcription_with_ollama(
        transcription, 
        model_name=model_name,  # Specify the model
        prompt=custom_prompt  # Pass the custom prompt
    )
    
    # Append the summary to the summaries list
    # print(summary)
    summaries.append(summary)
    # time.sleep(1)  # Optional: Add a delay to prevent hitting the API too frequently

# Add the summaries and the model name (as a constant string) to the DataFrame
merged_df['summary'] = summaries
merged_df['model_name'] = model_name  # Add the same model name to all rows

# Display the first few rows of the merged DataFrame with summaries and model names
print(merged_df.head())


Summarizing: 100%|██████████| 2127/2127 [58:58<00:00,  1.66s/transcription] 

                                       transcription  \
0  ہیلو ، میں سکاٹ ریل ہوں اور expertvillage.com ...   
1  ایلیسن کلنگر: expertvillage.com کی جانب سے ، م...   
2  ٹھیک ہے تو یہ ہے بیومونٹ راگ ، پہلا نصف جو کہ ...   
3  ہم جھاڑو ہٹانے جا رہے ہیں اور آپ ہمیشہ اپنے کل...   
4  آپ جانتے ہیں ، پچھلے حصے میں ، ہم نے یہ تسلیم ...   

                                         description  \
0  بارہماسی باغبانی پر اس مفت ویڈیو کلپ میں اپنے ...   
1  کافی مفت یا گھنے شیشے اس مفت ویڈیو سیریز میں د...   
2  beaumont rag &#39;&#39; میں گانے کے حصے یا حصے...   
3  چہرے کی سکرب کریم کو آہستہ سے دھو لیں اور جلد ...   
4  جب ایک ہی درجہ بندی والے کارڈ ایک دوسرے کے قری...   

                                             summary model_name  
0  The transcription appears to be a conversation...     llama3  
1  The transcription is a video tutorial on cutti...     llama3  
2  The transcription appears to be in Urdu. Here'...     llama3  
3  The transcription is about a product that help...     llama

In [31]:
# Replace with your actual API key
api_key = "a01b7c3da4ad2a793195eb157d331de480c41245"

# Log in using the API key
wandb.login(key=api_key)

# Proceed with your WandB initialization and logging as before
wandb.init(project='Abstractive_Summarization', name='summarization_test_run_llama3')


# Log the DataFrame as a table on WandB
wandb_table = wandb.Table(dataframe=merged_df)

# Log the table to WandB
wandb.log({"summarized_data": wandb_table})

# Finish the WandB run
wandb.finish()

# Display the first few rows of the merged DataFrame with summaries
print(merged_df.head())

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


                                       transcription  \
0  ہیلو ، میں سکاٹ ریل ہوں اور expertvillage.com ...   
1  ایلیسن کلنگر: expertvillage.com کی جانب سے ، م...   
2  ٹھیک ہے تو یہ ہے بیومونٹ راگ ، پہلا نصف جو کہ ...   
3  ہم جھاڑو ہٹانے جا رہے ہیں اور آپ ہمیشہ اپنے کل...   
4  آپ جانتے ہیں ، پچھلے حصے میں ، ہم نے یہ تسلیم ...   

                                         description  \
0  بارہماسی باغبانی پر اس مفت ویڈیو کلپ میں اپنے ...   
1  کافی مفت یا گھنے شیشے اس مفت ویڈیو سیریز میں د...   
2  beaumont rag &#39;&#39; میں گانے کے حصے یا حصے...   
3  چہرے کی سکرب کریم کو آہستہ سے دھو لیں اور جلد ...   
4  جب ایک ہی درجہ بندی والے کارڈ ایک دوسرے کے قری...   

                                             summary model_name  
0  The transcription appears to be a conversation...     llama3  
1  The transcription is a video tutorial on cutti...     llama3  
2  The transcription appears to be in Urdu. Here'...     llama3  
3  The transcription is about a product that help...     llama